# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the [FAIR² Mental Health Survey](https://sen.science/doi/10.71728/senscience.vcs2-05nj) dataset using the `mlcroissant` library. The dataset contains demographic and mental health survey data collected in Kilifi County, Kenya.

### Dataset Source
The dataset is described via a Croissant schema and is AI-ready.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and explore core properties with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Explore available record sets, their fields, and the `@id` for each entity. The Croissant schema strictly uses `@id` to uniquely reference entities:

In [ ]:
# Display all record sets and their fields with their @id
record_sets_info = []
for rs in metadata.record_sets:
    print(f"RecordSet name: {rs.name}  |  @id: {rs.id}")
    field_ids = [field.id for field in rs.fields]
    print(f"  Fields (@id): {field_ids}")
    record_sets_info.append({"name": rs.name, "id": rs.id, "field_ids": field_ids})
    print()
if not record_sets_info:
    print("No record sets found. Please check the schema definition.")

## 3. Data Extraction
Load data from chosen record set(s) into a DataFrame for exploration.

- Use the `@id` for record sets and fields obtained in the previous step.
- Note: The Kilifi dataset defines the survey responses in the primary record set. Adjust `record_sets_to_load` variable as needed.

In [ ]:
# Prepare record set IDs as discovered above
# Example: survey_records_id = 'cr:RecordSet:kilifi_survey'  # replace with the actual @id from previous output
record_sets_to_load = [rs["id"] for rs in record_sets_info if rs["field_ids"]]  # load all non-empty record sets

dataframes = {}
for record_set_id in record_sets_to_load:
    print(f"Loading record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Number of records: {len(df)}\n")

# For demonstration, pick the first record set for further analysis
if record_sets_to_load:
    chosen_record_set_id = record_sets_to_load[0]
    display_cols = dataframes[chosen_record_set_id].columns.tolist()
    print(f"Columns for '{chosen_record_set_id}': {display_cols}")
    dataframes[chosen_record_set_id].head()
else:
    print('No record sets found to load!')

## 4. Exploratory Data Analysis (EDA)
Let's perform some common data processing operations:

- **Filtering records:** Select records meeting a condition.
- **Normalizing numeric fields:** Transform distributions for analysis.
- **Grouping:** Aggregate results by key field(s).

**All field references use their `@id` as specified by the schema.**

In [ ]:
# Choose fields by their @id for numeric operations (e.g., GAD-7 score)
# You may need to adjust these based on the schema output above, e.g., 'gad7_total_score'
chosen_numeric_field = None
chosen_group_field = None
df = dataframes[chosen_record_set_id]
# Heuristically search for a relevant field
for col in df.columns:
    lower = col.lower()
    if "gad7" in lower or "phq9" in lower or "score" in lower:
        chosen_numeric_field = col
    if col in ["village", "@id:village", "village_id"] or "village" in lower:
        chosen_group_field = col
if chosen_numeric_field is None:
    # fallback: just pick the first numeric column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            chosen_numeric_field = col
            break

print(f"Using numeric field: {chosen_numeric_field}")
# Example: filter values above a threshold
threshold = 10
filtered_df = df[df[chosen_numeric_field] > threshold]
print(f"Filtered records with {chosen_numeric_field} > {threshold}:")
print(filtered_df.head())

filtered_df[f"{chosen_numeric_field}_normalized"] = (
    filtered_df[chosen_numeric_field] - filtered_df[chosen_numeric_field].mean()
) / filtered_df[chosen_numeric_field].std()
print(f"Normalized {chosen_numeric_field} for filtered records:")
print(filtered_df[[chosen_numeric_field, f"{chosen_numeric_field}_normalized"]].head())

if chosen_group_field and chosen_group_field in df.columns:
    grouped_df = filtered_df.groupby(chosen_group_field).mean(numeric_only=True)
    print(f"Grouped data by {chosen_group_field} (showing mean of numeric columns):")
    print(grouped_df.head())

## 5. Visualization
Let's visualize distributions and group statistics using `matplotlib` and `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of selected numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[chosen_numeric_field], kde=True)
plt.title(f'Distribution of {chosen_numeric_field}')
plt.xlabel(chosen_numeric_field)
plt.ylabel('Count')
plt.show()

# If grouping field exists, do boxplot
if chosen_group_field and chosen_group_field in df.columns:
    plt.figure(figsize=(10,4))
    sns.boxplot(data=df, x=chosen_group_field, y=chosen_numeric_field)
    plt.title(f'{chosen_numeric_field} by {chosen_group_field}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
- This notebook demonstrated loading Croissant metadata and records with consistent referencing by `@id` using `mlcroissant`.
- We explored, processed, and visualized key fields from the Kilifi mental health dataset.
- For advanced data science analysis, refer to the documentation at [mlcroissant.readthedocs.io](https://mlcroissant.readthedocs.io/) and tailor processing steps for your research questions.

<sup>FAIR²: Findable, Accessible, Interoperable, Reusable & Trustworthy, Transparent</sup>